# Four sources, one wind farm: from raw data to a common model behind an OGC API
### Companion to the data-integration exercise (≈ 45 min)

The SES notebooks need one thing from the wind side: *which farms, how big, where, when*. Four public sources answer that question for the Polish Baltic, and they do not agree:

| Source | What it publishes | How we got it |
|---|---|---|
| **OpenStreetMap** | every turbine position with model, rotor and hub height; farm areas | API 0.6 `map` call, 27 Oct 2026 |
| **EMODnet Human Activities** | one point per farm or licence area; polygons for planning zones | WFS `emodnet:windfarms`, `emodnet:windfarmspoly`, 30 Oct 2026 |
| **HubOcean Ocean Data Platform (ODP)** | a copy of the EMODnet layers | public vector tiles (the table API needs an account), 30 Oct 2026 |
| **National reporting** | project list with capacity, status and first-power year, no coordinates | gramwzielone.pl, 26 Aug 2026 |

This notebook does three things:
1. **shows the differences** between the raw sources, value by value;
2. **translates each source into a common model** — raw data → RDF through the source's **building block** → a common TBox (SPARQL CONSTRUCT) → RDF in the shape of the target block → GeoJSON in the target block's terms;
3. **puts both behind an OGC API – Features service** (pygeoapi): every raw source as it is, and harmonised collections translated *on request* from the raw files.

```
 raw data ──lift(block)──▶ RDF ──align──▶ common RDF ──validate──▶ ──resolve──▶ RDF ──lower(target block)──▶ GeoJSON / OGC API
           bblocks_ld        *.rq + crosswalk        SHACL        policy          bblocks_ld
```

## 0. Setup
The code is in three modules; the notebook only calls them:

| Module | What it holds |
|---|---|
| `bblocks_ld.py` | **generic, reusable**: `BuildingBlock` (context, schema, shapes from a repository or the register), `lift(data, block) → RDF`, `lower(RDF, block, type) → GeoJSON`, `construct`, `select`, `shacl_report` — rdflib, PyLD, shapely, geopandas, jsonschema, pyshacl |
| `windfarm_harmonise.py` | one function per pipeline step for this case, plus the pygeoapi provider |
| `../viz.py` | maps with geopandas/folium (`explore`, `plot_layers`) and charts; every function takes GeoDataFrames, GeoJSON, GeoParquet or DataFrames |

Every decision lives in a file under `config/multisource/`: `sources.json` (files, building block per source, IRI patterns), `tbox/` (common model), `alignments/*.rq` (source → common), `farm-crosswalk.json` (same-farm links), `shapes/` (SHACL), `projection.json` (which source wins), `projections/farms-pl.rq` (common → target block shape), `contexts/` (local contexts and patches).

In [1]:
import sys
sys.path.insert(0, "..")                      # viz.py lives next to the SES notebooks
import pandas as pd
import bblocks_ld as bl
import windfarm_harmonise as wh
import viz

pd.set_option("display.max_colwidth", 70)
sources = wh.load_sources()
raw = wh.read_all(sources)
wh.source_inventory(sources, raw)

,title,records,geometry,block / context,snapshot,licence
key,,,,,,
national,"National project list (gramwzielone.pl, 26 Aug 2026)",9,none,bblock:national-windfarm-source,2026-08-26,press report; facts compiled by the exercise
emodnet-wfs,"EMODnet Human Activities WFS emodnet:windfarms, Poland",26,Point,bblock:emodnet-compliant-windfarm,2026-10-30,EMODnet Open (attribution)
emodnet-wfs-zones,"EMODnet Human Activities WFS emodnet:windfarmspoly, Poland",7,MultiPolygon,bblock:emodnet-compliant-windfarm,2026-10-30,EMODnet Open (attribution)
odp-points,"HubOcean Ocean Data Platform: EMODnet wind farms (points), vector ...",13,Point,bblock:emodnet-compliant-windfarm,2026-10-30,EMODnet Open (attribution)
odp-zones,"HubOcean Ocean Data Platform: EMODnet wind farms (polygons), vecto...",7,MultiPolygon,bblock:emodnet-compliant-windfarm,2026-10-30,EMODnet Open (attribution)
osm,"OpenStreetMap, offshore Polish Baltic (API 0.6 map call)",292,"Point, Polygon, none",osm-raw.jsonld,2026-10-27,"ODbL 1.0, © OpenStreetMap contributors"


The files in `../data/` are the responses exactly as received (checked byte for byte against the browser download with SHA-256). The ODP file is the only exception: vector tiles are binary, so it holds the decoded tile features with coordinates converted from tile pixels to longitude and latitude.

## 1. How the sources differ (10 min)
### 1a. Different things on the map
Toggle the layers: OSM describes **turbines** and **farm areas**; EMODnet describes **farms or licence areas as points** and **planning zones as polygons**; ODP republishes an **older edition** of EMODnet (polygons cut at tile edges); the national list has **no geometry at all**.

In [2]:
STYLE = {"OSM farm areas": {"color": "#4338ca", "style_kwds": {"fillOpacity": .2}},
         "OSM turbines": {"color": "#1e3a8a", "marker_kwds": {"radius": 2}},
         "EMODnet zones (WFS)": {"color": "#c2410c", "style_kwds": {"fillOpacity": .15}},
         "EMODnet points (WFS)": {"color": "#c2410c", "marker_kwds": {"radius": 6}},
         "ODP zones (tiles)": {"color": "#15803d", "style_kwds": {"fillOpacity": .15}},
         "ODP points (tiles)": {"color": "#15803d", "marker_kwds": {"radius": 4}}}
viz.explore({name: {"data": data, **STYLE[name]} for name, data in wh.layers(raw).items()})

### 1b. The same farms, value by value
The comparison is read from the common graph built in part 3, so every value is already linked to its farm through the explicit crosswalk; here we only look at the raw values side by side.

In [3]:
raw_graphs = wh.uplift_all(sources, raw)
common, added = wh.align(sources, raw_graphs, wh.crosswalk_graph(raw))
zone_links = wh.link_zones(common, raw)
national_projects = [f["properties"]["farm"] for f in raw["national"]["features"]]
wh.compare_sources(common, names=national_projects)

,MW emodnet-wfs,MW national,MW odp-points,MW osm,turbines emodnet-wfs,turbines odp-points,turbines osm,status emodnet-wfs,status national,status odp-points,status osm,year emodnet-wfs,year national,year odp-points,year osm,turbines mapped (OSM)
farm,,,,,,,,,,,,,,,,
MFW Baltic Power,1140.0,1200.0,1140.0,1200.0,76.0,76.0,NaN,Construction,Construction,Construction,,2026,2026,,,76.0
Baltica 2,1498.0,1500.0,1498.0,NaN,NaN,NaN,NaN,Planned,Construction,Planned,Construction,2027,2027,,,107.0
Baltyk 2 + Baltyk 3,1440.0,1440.0,1440.0,720.0,NaN,NaN,50.0,Planned,Construction,Planned,Construction,2028,2028,,2027,100.0
BC-Wind,500.0,390.0,NaN,NaN,NaN,NaN,NaN,Planned,Construction,NaN,NaN,2028,2028,NaN,NaN,NaN
Baltica 3,1045.5,1000.0,1045.5,NaN,NaN,NaN,NaN,Planned,Planned,Planned,,2030,2030,,,NaN
MFW Baltyk 1,1560.0,1560.0,1560.0,NaN,260.0,260.0,NaN,Planned,Planned,Planned,NaN,2028,2032,,NaN,NaN
Baltica 9+,1325.0,1300.0,350.0,NaN,NaN,NaN,NaN,Planned,Planned,Planned,NaN,2030,2032,,NaN,NaN
Baltic East,966.0,900.0,NaN,NaN,NaN,NaN,NaN,Planned,Planned,NaN,NaN,,2032,NaN,NaN,NaN
Baltic West,NaN,4100.0,NaN,NaN,NaN,NaN,NaN,NaN,Planned,NaN,NaN,NaN,2035,NaN,NaN,NaN


In [4]:
wh.turbine_to_point_distance(common)

,EMODnet point → OSM turbine centroid (km)
farm,
baltica-2,0.17
baltyk-2-plus-baltyk-3,0.73
mfw-baltic-power,0.31


> **Read the table.** Nothing here is an error by itself; each source answers a slightly different question.
> * **Capacity.** Baltic Power: 1200 MW nationally and in the OSM plant relation, 1140 MW in EMODnet, ODP and the 76 × 15 MW turbines. Baltica 9+ is 1300 MW nationally, 1325 MW as two EMODnet sites, but only 350 MW in ODP — whose older edition lacks Baltica 9. BC-Wind: 390 MW vs 500 MW (two EMODnet licence areas).
> * **Status.** Bałtyk 2 + 3 and Baltica 2 are under construction nationally and in OSM, but *Planned* in EMODnet: status ages faster than an annual update.
> * **Year.** Bałtyk 1 first power: 2032 nationally (after the Dec 2025 auction), 2028 in EMODnet.
> * **Location.** EMODnet's single point per farm lies only 0.2–0.7 km from the centroid of the mapped turbines — but one point cannot describe a farm of 100–200 km² for wake, fishing-displacement or habitat-overlap analysis.
> * **Unit of record.** One national project = two EMODnet points (Bałtyk 2 + 3, BC-Wind, Baltica 9+, Bałtyk 1); Baltic West has no EMODnet counterpart at all.

### 1c. EMODnet and its copy on ODP

In [5]:
wh.point_offsets(raw, "emodnet-wfs", "odp-points")

,in odp-points,updateyear emodnet-wfs,updateyear odp-points,MW emodnet-wfs,MW odp-points,offset (m)
name,,,,,,
Baltic East,False,2026,NaN,966.0,NaN,NaN
Energa MFW 1 Offshore wind farm 1,False,2026,NaN,900.0,NaN,NaN
Freeboard I Offshore wind farm 1,False,2026,NaN,1500.0,NaN,NaN
Orlen Neptun IV Offshore wind farm 1,False,2026,NaN,1200.0,NaN,NaN
Baltica V Offshore wind farm 1,False,2026,NaN,555.0,NaN,NaN
Orlen Neptun III Offshore wind farm 1,False,2026,NaN,1200.0,NaN,NaN
Energa MFW 2 Offshore wind farm 1,False,2026,NaN,800.0,NaN,NaN
Baltica IV Offshore wind farm 1,False,2026,NaN,990.0,NaN,NaN
Orlen Neptun VIII Offshore wind farm 1,False,2026,NaN,1000.0,NaN,NaN


> The ODP copy is an **earlier edition** (`updateyear` 2024–2025, 13 Polish records) without the year, installation type and notes fields; its coordinates are shifted by about 10 m because vector tiles quantise them to a 4096-pixel grid per tile. A copy is a new dataset with its own edition and processing history — a twin must know which one it read.

## 2. Lift: each source in its own terms (7 min)
`bl.lift(data, block)` changes no value. Each raw feature gets a global identifier and is read with **its source's building block**: the published `emodnet-compliant-windfarm` and `national-windfarm-source` blocks, the EMODnet block again for the ODP copy (it is EMODnet data), and a local context for OSM tag keys. Geometry becomes GeoSPARQL WKT.

In [6]:
pd.DataFrame([{"source": s["key"], "building block / context": repr(wh.block(s["block"])),
               "triples": len(raw_graphs[s["key"]])} for s in sources]).set_index("source")

,building block / context,triples
source,,
national,<BuildingBlock national-windfarm-source from /home/claude/ogcincub...,72
emodnet-wfs,<BuildingBlock emodnet-compliant-windfarm from /home/claude/ogcinc...,295
emodnet-wfs-zones,<BuildingBlock emodnet-compliant-windfarm from /home/claude/ogcinc...,82
odp-points,<BuildingBlock emodnet-compliant-windfarm from /home/claude/ogcinc...,133
odp-zones,<BuildingBlock emodnet-compliant-windfarm from /home/claude/ogcinc...,93
osm,<BuildingBlock osm-raw.jsonld from local context>,4805


In [7]:
odp_baltic_power = [f for f in raw["odp-points"]["features"] if f["id"] == "baltic-power"]
emodnet_block = wh.block("bblock:emodnet-compliant-windfarm")
print(bl.lift(odp_baltic_power, emodnet_block, id_fn=lambda f: "https://example.org/odp/" + f["id"]).serialize(format="turtle"))

@prefix geo: <http://www.opengis.net/ont/geosparql#> .
@prefix geojson: <https://purl.org/geojson/vocab#> .
@prefix schema: <https://schema.org/> .
@prefix seadots: <https://w3id.org/ogc/hosted/seadots/emodnet-compliant-windfarm#> .
@prefix status: <https://w3id.org/ogc/hosted/seadots/emodnet/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/odp/baltic-power> a geojson:Feature ;
    geo:hasGeometry [ geo:asWKT "POINT (17.64988 55.05006)"^^geo:wktLiteral ] ;
    geojson:properties [ schema:addressCountry status:Poland ;
            seadots:distanceToCoast 2.842691e+04 ;
            seadots:nTurbines 76 ;
            seadots:powerMW 1140 ;
            seadots:updateYear "2025" ;
            status: status:Construction ;
            status:name "Baltic Power" ] .




The ODP record now speaks the EMODnet block's vocabulary (`em:powerMW`, the status as a `seadots-emodnet:` concept): **the building block does the first half of the harmonisation**.

## 3. Align: one common TBox (8 min)
The common model (`tbox/offshore-wind-common.ttl`) reuses every term the blocks define — `em:powerMW`, `em:nTurbines`, `em:year`, the `seadots-emodnet:` status concepts, the `ses:` provenance terms — and adds only what no block has yet: turbines, planning zones and **source records**. A source record is what one dataset says about one farm; records from different sources stay side by side and are linked to the real-world farm with `owc:describes`. One CONSTRUCT per source *shape* does the translation; the same query serves the EMODnet WFS and its ODP copy:

In [8]:
print((wh.CONFIG / "alignments/emodnet-points.rq").read_text())
print("triples added per source:", added, "| farm-in-zone links:", zone_links, "| common graph:", len(common))

# EMODnet wind-farm points (emodnet-compliant-windfarm block) -> common records.
# Used unchanged for the EMODnet WFS and for the HubOcean ODP copy: both carry the EMODnet schema.
CONSTRUCT {
  ?rec a owc:SourceRecord ; owc:recordKind owc:WindFarm ; owc:sourceKey ?key ;
       schema:name ?name ; em:powerMW ?mw ; em:nTurbines ?n ; owc:status ?status ; owc:sourceStatusText ?statusText ;
       em:year ?year ; em:updateYear ?updateyear ; em:distanceToCoast ?dist ; schema:additionalType ?inst ;
       owc:locationKind "point" ; geo:hasGeometry ?geom ;
       dcterms:source ?dataset ; ses:sourceSnapshotDate ?snapshot ; prov:wasDerivedFrom ?f .
  ?geom geo:asWKT ?wkt .
}
WHERE {
  ?f geojson:properties ?p ; geo:hasGeometry ?geom . ?geom geo:asWKT ?wkt .
  ?p sem:name ?name .
  OPTIONAL { ?p em:powerMW ?mw0 } OPTIONAL { ?p em:nTurbines ?n0 }
  OPTIONAL { ?p sem: ?status . BIND(STRAFTER(STR(?status), STR(sem:)) AS ?statusText) }
  OPTIONAL { ?p em:year ?year } OPTIONAL { ?p em:updateYear ?upd

In [9]:
records = wh.records(common)
records[records.target == "urn:baltic-ses:farm:mfw-baltic-power"][["source", "name", "power_mw", "n_turbines", "status", "year", "derived"]]

,source,name,power_mw,n_turbines,status,year,derived
1,national,MFW Baltic Power,1200,NaN,https://w3id.org/ogc/hosted/seadots/emodnet/Construction,2026,urn:baltic-ses:national:mfw-baltic-power
22,emodnet-wfs,Baltic Power,1140.0,76.0,https://w3id.org/ogc/hosted/seadots/emodnet/Construction,2026,https://ows.emodnet-humanactivities.eu/wfs?service=WFS&version=2.0...
47,odp-points,Baltic Power,1140,76.0,https://w3id.org/ogc/hosted/seadots/emodnet/Construction,NaN,https://app.hubocean.earth/catalog/dataset/421294d9-696e-47a8-9f23...
63,osm,Baltic Power,1200,NaN,NaN,NaN,https://www.openstreetmap.org/relation/21402440
65,osm,Baltic Power,None,NaN,NaN,NaN,https://www.openstreetmap.org/way/1532855580


In [10]:
bl.select(common, '''SELECT ?model (COUNT(?t) AS ?turbines) (SAMPLE(?rotor) AS ?rotor_m) (SAMPLE(?hub) AS ?hub_m)
                     (SAMPLE(?rated) AS ?rated_mw) WHERE {
                       ?t owc:partOfFarm ?farm ; owc:turbineModel ?model .
                       OPTIONAL { ?t owc:rotorDiameterM ?rotor } OPTIONAL { ?t owc:hubHeightM ?hub }
                       OPTIONAL { ?t owc:ratedPowerMW ?rated } } GROUP BY ?model''', init_ns=wh.NS)

,model,turbines,rotor_m,hub_m,rated_mw
0,SG 14-236 DD,100,236,None,None
1,V236-15.0 MW,76,236,141,15
2,SG 14-222 DD,107,222,None,14


**The crosswalk** is the one place where a human decides that two records are the same farm: national ↔ EMODnet links from `location-mapping.json`, ODP records by their identical EMODnet name, OSM elements by id and OSM turbines by a rule (relation membership, model, `ref` prefix). Planning zones are linked spatially with a geopandas `sjoin`.

> **Watch out for identifiers.** A first version made farm IRIs from slugs of names: "Baltica 2" and "Baltica 2+" both became `baltica-2`, silently merging two EMODnet records into one 1708 MW farm. SHACL caught it as a capacity disagreement. Identifiers are design decisions, not formatting.

## 4. Validate (5 min)
SHACL shapes check every translated record regardless of its source (identifiers and provenance present, capacity a positive MW value below 6 GW, status from the EMODnet vocabulary, each turbine in exactly one farm). Two SPARQL-based shapes compare sources: capacity disagreement above 5 % and reported vs mapped turbine numbers are **warnings**.

In [11]:
conforms, report = wh.validate(common)
print("conforms (no violations):", conforms)
display(report.groupby(["severity", "message"]).size().rename("count").reset_index())
report[report.severity == "Warning"][["focus", "message"]]

conforms (no violations): True


,severity,message,count
0,Info,No status that maps to the EMODnet vocabulary.,3
1,Info,Turbine has no hub height in the source.,207
2,Info,Turbine has no rated power in the source; it must come from the tu...,100
3,Warning,"Reported number of turbines (50, osm) differs from turbines mapped...",1
4,Warning,Sources disagree on capacity by more than 5 %: 1140 MW (odp-points...,1
5,Warning,Sources disagree on capacity by more than 5 %: 1140 MW (odp-points...,1
6,Warning,Sources disagree on capacity by more than 5 %: 1140.0 MW (emodnet-...,1
7,Warning,Sources disagree on capacity by more than 5 %: 1140.0 MW (emodnet-...,1
8,Warning,Sources disagree on capacity by more than 5 %: 350 MW (odp-points)...,1
9,Warning,Sources disagree on capacity by more than 5 %: 350 MW (odp-points)...,1


,focus,message
310,baltic-east,Sources disagree on capacity by more than 5 %: 900 MW (national) v...
311,baltica-9-plus,Sources disagree on capacity by more than 5 %: 350 MW (odp-points)...
312,baltica-9-plus,Sources disagree on capacity by more than 5 %: 350 MW (odp-points)...
313,baltyk-2-plus-baltyk-3,"Reported number of turbines (50, osm) differs from turbines mapped..."
314,baltyk-2-plus-baltyk-3,Sources disagree on capacity by more than 5 %: 720 MW (osm) vs 144...
315,baltyk-2-plus-baltyk-3,Sources disagree on capacity by more than 5 %: 720 MW (osm) vs 144...
316,baltyk-2-plus-baltyk-3,Sources disagree on capacity by more than 5 %: 720 MW (osm) vs 144...
317,bc-wind,Sources disagree on capacity by more than 5 %: 390 MW (national) v...
318,mfw-baltic-power,Sources disagree on capacity by more than 5 %: 1140 MW (odp-points...
319,mfw-baltic-power,Sources disagree on capacity by more than 5 %: 1140 MW (odp-points...


> **Read the warnings.** Most are the disagreements of part 1. Two only become visible after linking: OSM's *MFW Bałtyk II* area says 720 MW and 50 turbines, while the national project *Bałtyk 2 + 3* is 1440 MW with 100 mapped turbines. Both are right — the OSM area is **part of** the project. The crosswalk has one kind of link (`owc:describes`); a production model would also need *part of*, so a projection can sum parts instead of comparing them.

## 5. Resolve and lower onto the target building block (5 min)
Two steps, both producing standard outputs:
1. `wh.resolve(common, ...)` applies `projection.json` — which source wins for each property (national list for capacity, status and year; OSM for the turbine count and location; EMODnet for distance to coast) — and writes the result **as RDF**, with the source of every choice (`owc:valueSource`) and the provenance fields of the Polish profile.
2. `wh.to_profile(resolved)` maps that RDF to the RDF shape of the target block with one CONSTRUCT (`projections/farms-pl.rq`), then calls the generic `bl.lower(graph, block, type)`: JSON-LD framing with the block's own context produces GeoJSON in the block's property names, checked against the block's JSON Schema.

In [12]:
resolved = wh.resolve(common, sources)
collections = wh.to_profile(resolved)
errors = wh.validate_profile(collections["farms-pl"])
print("farms valid against emodnet-compliant-windfarm-pl:", sum(not e for e in errors.values()), "of", len(errors))
viz.to_gdf(collections["farms-pl"])[["name", "power_mw", "n_turbines", "status", "year", "source_type",
                                     "location_status", "location_method", "notes"]]

farms valid against emodnet-compliant-windfarm-pl: 23 of 23


,name,power_mw,n_turbines,status,year,source_type,location_status,location_method,notes
0,B-Wind,200.0,NaN,Planned,NaN,emodnet,source-provided,emodnet-wfs point(s),NaN
1,Baltic East,900.0,NaN,Planned,2032,national,mapped,emodnet-wfs point(s),values merged from several sources: updateyear from emodnet-wfs; d...
2,Baltic West,4100.0,NaN,Planned,2035,national,anticipated,Approximate representative point inferred from ORLEN Neptun's desc...,values merged from several sources: geometry from anticipated (loc...
3,Baltica 1,900.0,NaN,Planned,2030,emodnet,source-provided,emodnet-wfs point(s),NaN
4,Baltica 2,1500.0,107.0,Construction,2027,national,mapped,centroid of 107 OpenStreetMap turbine positions,values merged from several sources: n_turbines from osm-count; upd...
5,Baltica 2+,210.0,NaN,Planned,NaN,emodnet,source-provided,emodnet-wfs point(s),NaN
6,Baltica 3,1000.0,NaN,Planned,2030,national,mapped,emodnet-wfs point(s),values merged from several sources: updateyear from emodnet-wfs; d...
7,Baltica 9+,1300.0,NaN,Planned,2032,national,mapped,emodnet-wfs (centroid of 2 points) point(s),values merged from several sources: updateyear from emodnet-wfs; d...
8,Baltica IV Offshore wind farm 1,990.0,NaN,Planned,NaN,emodnet,source-provided,emodnet-wfs point(s),NaN
9,Baltica V Offshore wind farm 1,555.0,NaN,Planned,NaN,emodnet,source-provided,emodnet-wfs point(s),NaN


In [13]:
wh.value_choices(collections["farms-common"]).loc[national_projects, ["power_mw", "n_turbines", "status", "year", "geometry"]].fillna("—")

,power_mw,n_turbines,status,year,geometry
MFW Baltic Power,national,osm-count,national,national,osm-turbine-centroid
Baltica 2,national,osm-count,national,national,osm-turbine-centroid
Baltyk 2 + Baltyk 3,national,osm-count,national,national,osm-turbine-centroid
BC-Wind,national,—,national,national,emodnet-wfs (centroid of 2 points)
Baltica 3,national,—,national,national,emodnet-wfs
MFW Baltyk 1,national,emodnet-wfs,national,national,emodnet-wfs (centroid of 2 points)
Baltica 9+,national,—,national,national,emodnet-wfs (centroid of 2 points)
Baltic East,national,—,national,national,emodnet-wfs
Baltic West,national,—,national,national,anticipated (location-mapping.json)


In [14]:
viz.explore({
    "planning zones (harmonised)": {"data": collections["zones"], "color": "#c2410c", "style_kwds": {"fillOpacity": .12},
                                    "tooltip": ["name", "area_km2", "status", "farms"]},
    "turbines (harmonised)": {"data": collections["turbines"], "column": "model", "cmap": "Set1", "marker_kwds": {"radius": 2}, "legend": False,
                              "tooltip": ["ref", "model", "rotor_diameter_m", "hub_height_m", "farm"]},
    "farms (emodnet-compliant-windfarm-pl)": {"data": collections["farms-pl"], "column": "status", "cmap": "Dark2",
                                             "marker_kwds": {"radius": 8},
                                             "tooltip": ["name", "power_mw", "n_turbines", "status", "year", "location_method"]}})

> **A finding about the target block.** Lowering through the published `emodnet-compliant-windfarm-pl` context needs one local patch (`contexts/pl-status-terms.jsonld`). The block's context sets `@vocab` to its own namespace and types `status` as `@vocab`, so a value such as `"Construction"` expands to `…emodnet-compliant-windfarm-pl#Construction` instead of the EMODnet concept `seadots-emodnet:Construction`. The base block does not have this problem because its `@vocab` is the EMODnet namespace. The fix belongs in the block: define the five status terms in its context. The base block also maps `name` differently (`seadots-emodnet:name`, via `@vocab`) from the profile (`schema:name`), so base and profile records describe the same field with two IRIs.

## 6. Serve: OGC API – Features (5 min)
`wh.write_service` generates a pygeoapi configuration with ten collections:
* six **raw** collections served exactly as received; with `f=jsonld` they carry their building-block context, i.e. the lift step;
* four **harmonised** collections — `farms-pl`, `farms-common`, `turbines`, `zones` — served by `windfarm_harmonise.HarmonisedProvider`, which runs lift → align → resolve → lower on the raw files when a request arrives (cached until a file changes) and honours `bbox`.

The service uses the environment of the first exercise (`./setup_pygeoapi.sh`, which now also installs the pipeline's libraries) and listens on port 5001.

In [15]:
import os
SERVICE = wh.ROOT / "service-multisource"
wh.write_service(SERVICE, {"sources": sources, "raw": raw, "collections": collections}, port=5001)
server = wh.start_service(SERVICE, os.environ.get("PYGEOAPI", wh.ROOT / ".venv/bin/pygeoapi"))
print("pygeoapi running at http://localhost:5001" if server else
      "pygeoapi not installed: run ./setup_pygeoapi.sh, then ./service-multisource/run.sh")

pygeoapi running at http://localhost:5001


In [16]:
if server:
    print([c["id"] for c in wh.api_get("/collections?f=json")["collections"]])
    bbox = "/items?f=json&bbox=17.55,54.97,17.80,55.12&limit=500"
    print("turbines in the Baltic Power bbox:", wh.api_get("/collections/turbines" + bbox)["numberMatched"],
          "| raw OSM in the same bbox (built-in GeoJSON provider ignores bbox):", wh.api_get("/collections/raw-osm" + bbox)["numberMatched"])
    side_by_side = pd.DataFrame({"raw-emodnet-points": pd.Series(wh.api_get("/collections/raw-emodnet-points/items/windfarms.268?f=json")["properties"]),
                                 "farms-pl": pd.Series(wh.api_get("/collections/farms-pl/items/mfw-baltic-power?f=json")["properties"])})
    display(side_by_side.fillna("—"))
    display(viz.explore({"turbines from the API (bbox query)": {"data": wh.api_get("/collections/turbines" + bbox),
                                                        "tooltip": ["ref", "model", "hub_height_m"]}}))

['raw-national', 'raw-emodnet-points', 'raw-emodnet-zones', 'raw-odp-points', 'raw-odp-zones', 'raw-osm', 'farms-pl', 'farms-common', 'turbines', 'zones']


turbines in the Baltic Power bbox: 76 | raw OSM in the same bbox (built-in GeoJSON provider ignores bbox): 292


,raw-emodnet-points,farms-pl
country,Poland,Poland
developer,—,Orlen; Northland Power
dist_coast,28426.906965,28426.906965
location_emodnet_ids,—,[https://ows.emodnet-humanactivities.eu/wfs?service=WFS&version=2....
location_method,—,centroid of 76 OpenStreetMap turbine positions
location_reference,—,https://www.openstreetmap.org/relation/21402440
location_status,—,mapped
n_turbines,76,76
name,Baltic Power,MFW Baltic Power
notes,—,values merged from several sources: n_turbines from osm-count; upd...


In [17]:
if server:
    server.terminate(); server.wait(timeout=10)
    print("service stopped; start it again with ./service-multisource/run.sh")

service stopped; start it again with ./service-multisource/run.sh


## Your turn
1. **Change the policy.** In `config/multisource/projection.json` put `emodnet-wfs` before `national` for `power_mw`. Re-run part 5 (or restart the service). Which farms change? Which answer would you give the SES model, and why?
2. **Another target block.** Lower the same resolved graph to the base block: `bl.lower(pl_graph, wh.block("bblock:emodnet-compliant-windfarm"), wh.GEOJSON.Feature)` with `pl_graph` from `bl.construct(resolved, ...)`. Which fields disappear, and which come out under different names?
3. **A new source.** HELCOM's maritime-spatial-planning layer has 13 Polish "Offshore wind farms" polygons. Add it to `sources.json` with the EMODnet zone query. What did you have to write — a context, a query, crosswalk entries — and what did you reuse?
4. **Tighten a rule.** Turn the capacity-agreement shape into a violation. Which farms would now block a release, and is that what you want for a twin that updates nightly?

## What this companion shows — and what it still assumes
| | Status |
|---|---|
| Raw values, geometries, identifiers | **DATA**, as fetched; ODP decoded from vector tiles |
| Source semantics | published building-block contexts (EMODnet, national); a local context for OSM |
| Common TBox | reuses `em:`, `seadots-emodnet:`, `ses:`; the `owc:` terms are local — a candidate for a turbine-level building block |
| Same-farm links | **ASSUMPTION**, explicit in the crosswalk files; zone membership by a point-in-polygon rule |
| Which value wins | **POLICY**, explicit in `projection.json`; every choice is reported in `farms-common` |
| Target profile | the published `emodnet-compliant-windfarm-pl` block, with one local context patch for its status terms |
| Not covered | substations, the HELCOM MSP layer, temporal versioning of editions |

### Sources
* OpenStreetMap contributors, © ODbL 1.0: https://www.openstreetmap.org/copyright
* EMODnet Human Activities, wind farms: https://emodnet.ec.europa.eu/en/human-activities — WFS https://ows.emodnet-humanactivities.eu/wfs
* HubOcean Ocean Data Platform catalogue: https://app.hubocean.earth (datasets 421294d9-696e-47a8-9f23-0473d726abab and 8dbcac0b-bdce-4714-ab02-f2aee6c28e85)
* National project list: gramwzielone.pl, 26 Aug 2026 — https://www.gramwzielone.pl/?p=20373718
* SeaDOTs building blocks: https://ogcincubator.github.io/bblocks-seadots/ — pygeoapi: https://pygeoapi.io — PyLD: https://github.com/digitalbazaar/pyld